# Ewaluacja apelacji (na przykładzie baseline)

Dwa spojrzenia: **pokrycie** (czy porusza wymagane zagadnienia z `data/eval.json`)
oraz **jakość/forma** (jak u egzaminatora). Działa na zapisanej apelacji — najpierw
wygeneruj ją w `baseline_walkthrough.ipynb`.

## 0. Konfiguracja

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()
os.environ["LLM_MODEL"] = "gpt-5.4-mini"  # w notebookach tani model — to tylko demo
print("model:", os.environ["LLM_MODEL"], "| klucz z .env:", bool(os.environ.get("LLM_API_KEY")))

## 1. Wczytaj zapisaną apelację baseline

In [ ]:
from src.output import load_latest_appeal, latest_appeal_path

appeal = load_latest_appeal("baseline")
print("Wczytano:", latest_appeal_path("baseline"))
print(f"{len(appeal):,} znaków")

## 2. Pokrycie — czy porusza wymagane zagadnienia?

Sędzia-LLM ocenia każde zagadnienie z klucza (na bieżąco).

In [ ]:
from src.eval.coverage import evaluate

cov = evaluate(appeal, print_results=True)

## 3. Jakość / forma (ocena egzaminatora)

Ocena „miękka” wg kryteriów egzaminu radcowskiego (wymogi formalne / zastosowanie
przepisów / poprawność rozwiązania), skala 2–6.

> Sędzią jakości jest **mocny model** (`gpt-5.4`) — tani model nie wyłapuje błędów
> formalnych. To 1 wywołanie, niezależne od modelu z konfiguracji.

In [ ]:
from src.eval.quality import evaluate_quality

q = evaluate_quality(appeal)
print(q.reasoning, "\n")
print(f"  wymogi formalne:            {q.wymogi_formalne}/6")
print(f"  zastosowanie/interpretacja: {q.zastosowanie_i_interpretacja}/6")
print(f"  poprawność rozwiązania:     {q.poprawnosc_rozwiazania}/6")
print(f"  -- średnia:                 {q.srednia:.2f}/6")

## Podsumowanie — dwa spojrzenia

| miernik | pyta | łapie |
|---|---|---|
| **pokrycie** | czy poruszono wymagane zagadnienia? | pominięty zarzut |
| **jakość** | jak napisane (forma, argumentacja)? | słaba forma/warsztat |

Same punkty to nie wszystko — apelację ocenia się też za formę i jakość argumentacji.